# Spark

In [1]:
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 6g pyspark-shell"

## Example

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import monotonically_increasing_id

In [3]:
spark = SparkSession \
            .builder \
            .appName("test") \
            .getOrCreate()

26/06/25 00:44:08 WARN Utils: Your hostname, borja-dosuna-personal-computer resolves to a loopback address: 127.0.1.1; using 192.168.1.141 instead (on interface wlp6s0)
26/06/25 00:44:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/25 00:44:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df = spark.read.csv("./people.csv", header=True, sep=';')
df.show()

+-----+---+---------+
| name|age|      job|
+-----+---+---------+
|Jorge| 30|Developer|
|  Bob| 32|Developer|
+-----+---+---------+



In [5]:
df.count()

2

In [6]:
df.printSchema()

root
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- job: string (nullable = true)



In [7]:
df.select("name").show()
df.select(["name", "job"]).show()

+-----+
| name|
+-----+
|Jorge|
|  Bob|
+-----+

+-----+---------+
| name|      job|
+-----+---------+
|Jorge|Developer|
|  Bob|Developer|
+-----+---------+



In [8]:
df.filter(df['age'] > 31).show()

+----+---+---------+
|name|age|      job|
+----+---+---------+
| Bob| 32|Developer|
+----+---+---------+



In [9]:
df.withColumn('index', monotonically_increasing_id())

DataFrame[name: string, age: string, job: string, index: bigint]

## Learning on massive click logs with Spark

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructField, StringType, StructType, IntegerType
from pyspark.ml.feature import StringIndexer, VectorAssembler, OneHotEncoder
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [11]:
spark = SparkSession\
    .builder\
    .appName("CTR")\
    .getOrCreate()

26/06/25 00:44:14 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [12]:
schema = StructType([
    StructField("id", StringType(), True),
    StructField("click", IntegerType(), True),
    StructField("hour", IntegerType(), True),
    StructField("C1", StringType(), True),
    StructField("banner_pos", StringType(), True),
    StructField("site_id", StringType(), True),
    StructField("site_domain", StringType(), True),
    StructField("site_category", StringType(), True),
    StructField("app_id", StringType(), True),
    StructField("app_domain", StringType(), True),
    StructField("app_category", StringType(), True),
    StructField("device_id", StringType(), True),
    StructField("device_ip", StringType(), True),
    StructField("device_model", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("device_conn_type", StringType(), True),
    StructField("C14", StringType(), True),
    StructField("C15", StringType(), True),
    StructField("C16", StringType(), True),
    StructField("C17", StringType(), True),
    StructField("C18", StringType(), True),
    StructField("C19", StringType(), True),
    StructField("C20", StringType(), True),
    StructField("C21", StringType(), True),
])

In [13]:
df = spark.read.parquet("./train_parquet/")

In [14]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- click: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- C1: string (nullable = true)
 |-- banner_pos: string (nullable = true)
 |-- site_id: string (nullable = true)
 |-- site_domain: string (nullable = true)
 |-- site_category: string (nullable = true)
 |-- app_id: string (nullable = true)
 |-- app_domain: string (nullable = true)
 |-- app_category: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- device_ip: string (nullable = true)
 |-- device_model: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- device_conn_type: string (nullable = true)
 |-- C14: string (nullable = true)
 |-- C15: string (nullable = true)
 |-- C16: string (nullable = true)
 |-- C17: string (nullable = true)
 |-- C18: string (nullable = true)
 |-- C19: string (nullable = true)
 |-- C20: string (nullable = true)
 |-- C21: string (nullable = true)



In [15]:
df.count()

40428967

In [16]:
df = df.drop('id').drop('hour').drop('device_id').drop('device_ip')
df = df.withColumnRenamed("click", "label")

In [17]:
df.columns

['label',
 'C1',
 'banner_pos',
 'site_id',
 'site_domain',
 'site_category',
 'app_id',
 'app_domain',
 'app_category',
 'device_model',
 'device_type',
 'device_conn_type',
 'C14',
 'C15',
 'C16',
 'C17',
 'C18',
 'C19',
 'C20',
 'C21']

In [18]:
# Sample 10% to fit hardware constraints on a single machine
df = df.sample(fraction=0.1, seed=42)

In [19]:
df_train, df_test = df.randomSplit([0.7, 0.3], 42)

In [20]:
df_train.cache()

DataFrame[label: int, C1: string, banner_pos: string, site_id: string, site_domain: string, site_category: string, app_id: string, app_domain: string, app_category: string, device_model: string, device_type: string, device_conn_type: string, C14: string, C15: string, C16: string, C17: string, C18: string, C19: string, C20: string, C21: string]

In [21]:
df_train.count()

2830938

In [22]:
df_test.cache()

DataFrame[label: int, C1: string, banner_pos: string, site_id: string, site_domain: string, site_category: string, app_id: string, app_domain: string, app_category: string, device_model: string, device_type: string, device_conn_type: string, C14: string, C15: string, C16: string, C17: string, C18: string, C19: string, C20: string, C21: string]

In [23]:
df_test.count()

1212122

### One-hot encoding categorical features

In [24]:
categorical = df_train.columns
categorical.remove('label')
print(categorical)

['C1', 'banner_pos', 'site_id', 'site_domain', 'site_category', 'app_id', 'app_domain', 'app_category', 'device_model', 'device_type', 'device_conn_type', 'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21']


In [25]:
indexers = [
    StringIndexer(inputCol=c, outputCol="{0}_indexed".format(c)).setHandleInvalid("keep")
    for c in categorical
]

In [26]:
indexers

[StringIndexer_f5858070ca3e,
 StringIndexer_d5d8cf476d28,
 StringIndexer_0906bf9da3da,
 StringIndexer_1f4dca232cf8,
 StringIndexer_bc70a44a80fb,
 StringIndexer_14ee75181e21,
 StringIndexer_cbe713e9a8a1,
 StringIndexer_cf4dddb378e1,
 StringIndexer_70329c714105,
 StringIndexer_6e2b8068f2ce,
 StringIndexer_48d60677cd46,
 StringIndexer_6389c290ba15,
 StringIndexer_da192b2a5899,
 StringIndexer_e3f2529c48db,
 StringIndexer_df780226068e,
 StringIndexer_94f10dca2b33,
 StringIndexer_a2693177ff2b,
 StringIndexer_b456af225709,
 StringIndexer_5dbad00a5b58]

In [27]:
encoder = OneHotEncoder(
    inputCols=[indexer.getOutputCol() for indexer in indexers],
    outputCols=[
        "{0}_encoded".format(indexer.getOutputCol()) for indexer in indexers]
)
assembler = VectorAssembler(
    inputCols=encoder.getOutputCols(),
    outputCol="features"
)

In [28]:
stages = indexers + [encoder, assembler]
pipeline = Pipeline(stages=stages)


In [29]:
one_hot_encoder = pipeline.fit(df_train)

In [30]:
df_train_encoded = one_hot_encoder.transform(df_train)
df_train_encoded.show()

26/06/25 00:44:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/06/25 00:44:33 WARN DAGScheduler: Broadcasting large task binary with size 3.4 MiB


+-----+----+----------+--------+-----------+-------------+--------+----------+------------+------------+-----------+----------------+-----+---+---+----+---+---+------+---+----------+------------------+---------------+-------------------+---------------------+--------------+------------------+--------------------+--------------------+-------------------+------------------------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+------------------+--------------------------+-----------------------+---------------------------+-----------------------------+----------------------+--------------------------+----------------------------+----------------------------+---------------------------+--------------------------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+--------------------+
|label|  C1|banner_pos| site_id|site_domain|s

In [31]:
df_train_encoded = df_train_encoded.select(["label", "features"])
df_train_encoded.show()

+-----+--------------------+
|label|            features|
+-----+--------------------+
|    0|(21019,[5,7,2869,...|
|    0|(21019,[5,7,802,3...|
|    0|(21019,[5,7,802,3...|
|    0|(21019,[5,7,802,3...|
|    0|(21019,[5,7,1371,...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
+-----+--------------------+
only showing top 20 rows



In [32]:
df_train_encoded.cache()

DataFrame[label: int, features: vector]

In [33]:
df_train.unpersist()

DataFrame[label: int, C1: string, banner_pos: string, site_id: string, site_domain: string, site_category: string, app_id: string, app_domain: string, app_category: string, device_model: string, device_type: string, device_conn_type: string, C14: string, C15: string, C16: string, C17: string, C18: string, C19: string, C20: string, C21: string]

In [34]:
df_test_encoded = one_hot_encoder.transform(df_test)
df_test_encoded = df_test_encoded.select(["label", "features"])
df_test_encoded.show()

+-----+--------------------+
|label|            features|
+-----+--------------------+
|    0|(21019,[5,7,802,3...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
|    0|(21019,[5,7,14,32...|
+-----+--------------------+
only showing top 20 rows



In [35]:
df_test_encoded.cache()

DataFrame[label: int, features: vector]

In [36]:
df_test.unpersist()

DataFrame[label: int, C1: string, banner_pos: string, site_id: string, site_domain: string, site_category: string, app_id: string, app_domain: string, app_category: string, device_model: string, device_type: string, device_conn_type: string, C14: string, C15: string, C16: string, C17: string, C18: string, C19: string, C20: string, C21: string]

### Training and testing a logistic regression model

In [37]:
classifier = LogisticRegression(maxIter=20, regParam=0.000, elasticNetParam=0.000)

In [38]:
lr_model = classifier.fit(df_train_encoded)

26/06/25 00:44:35 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/06/25 00:44:48 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/06/25 00:44:49 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/06/25 00:44:50 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/06/25 00:44:52 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/06/25 00:44:53 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/06/25 00:44:53 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/06/25 00:44:53 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/06/25 00:44:53 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/06/25 00:44:53 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/06/25 00:44:54 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/06/25 00:44:54 WARN DAGScheduler: Broadcasting larg

In [39]:
df_train_encoded.unpersist()

DataFrame[label: int, features: vector]

In [40]:
predictions = lr_model.transform(df_test_encoded)

In [41]:
df_test_encoded.unpersist()

DataFrame[label: int, features: vector]

In [42]:
predictions.cache()

DataFrame[label: int, features: vector, rawPrediction: vector, probability: vector, prediction: double]

In [43]:
predictions.show()

26/06/25 00:45:02 WARN DAGScheduler: Broadcasting large task binary with size 2.8 MiB


+-----+--------------------+--------------------+--------------------+----------+
|label|            features|       rawPrediction|         probability|prediction|
+-----+--------------------+--------------------+--------------------+----------+
|    0|(21019,[5,7,802,3...|[4.31516851134509...|[0.98681195173661...|       0.0|
|    0|(21019,[5,7,14,32...|[5.82800492090206...|[0.99706469684230...|       0.0|
|    0|(21019,[5,7,14,32...|[5.82800492090206...|[0.99706469684230...|       0.0|
|    0|(21019,[5,7,14,32...|[5.58057458580633...|[0.99624376402757...|       0.0|
|    0|(21019,[5,7,14,32...|[5.84519783982263...|[0.99711458754270...|       0.0|
|    0|(21019,[5,7,14,32...|[5.84519783982263...|[0.99711458754270...|       0.0|
|    0|(21019,[5,7,14,32...|[5.84519783982263...|[0.99711458754270...|       0.0|
|    0|(21019,[5,7,14,32...|[5.84519783982263...|[0.99711458754270...|       0.0|
|    0|(21019,[5,7,14,32...|[5.84519783982263...|[0.99711458754270...|       0.0|
|    0|(21019,[5

26/06/25 00:45:10 WARN DAGScheduler: Broadcasting large task binary with size 2.8 MiB


In [44]:
ev = BinaryClassificationEvaluator(rawPredictionCol = "rawPrediction", metricName = "areaUnderROC")

In [45]:
print(ev.evaluate(predictions))

26/06/25 00:45:10 WARN DAGScheduler: Broadcasting large task binary with size 2.8 MiB


0.7449368900796921


In [46]:
spark.stop()

## Feature engineering on categorical variables with Spark

### Feature hashing

In [47]:
from pyspark.ml.feature import FeatureHasher

In [48]:
spark = SparkSession\
    .builder\
    .appName("CTR")\
    .getOrCreate()

In [49]:
schema = StructType([
    StructField("id", StringType(), True),
    StructField("click", IntegerType(), True),
    StructField("hour", IntegerType(), True),
    StructField("C1", StringType(), True),
    StructField("banner_pos", StringType(), True),
    StructField("site_id", StringType(), True),
    StructField("site_domain", StringType(), True),
    StructField("site_category", StringType(), True),
    StructField("app_id", StringType(), True),
    StructField("app_domain", StringType(), True),
    StructField("app_category", StringType(), True),
    StructField("device_id", StringType(), True),
    StructField("device_ip", StringType(), True),
    StructField("device_model", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("device_conn_type", StringType(), True),
    StructField("C14", StringType(), True),
    StructField("C15", StringType(), True),
    StructField("C16", StringType(), True),
    StructField("C17", StringType(), True),
    StructField("C18", StringType(), True),
    StructField("C19", StringType(), True),
    StructField("C20", StringType(), True),
    StructField("C21", StringType(), True),
])

In [50]:
df = spark.read.parquet("./train_parquet/")

In [51]:
df = df.drop('id').drop('hour').drop('device_id').drop('device_ip')
df = df.withColumnRenamed("click", "label")
df = df.sample(fraction=0.1, seed=42)
df_train, df_test = df.randomSplit([0.7, 0.3], 42)
df_train.cache()
df_test.cache()

DataFrame[label: int, C1: string, banner_pos: string, site_id: string, site_domain: string, site_category: string, app_id: string, app_domain: string, app_category: string, device_model: string, device_type: string, device_conn_type: string, C14: string, C15: string, C16: string, C17: string, C18: string, C19: string, C20: string, C21: string]

In [52]:
categorical = df_train.columns
categorical.remove('label')
print(categorical)

['C1', 'banner_pos', 'site_id', 'site_domain', 'site_category', 'app_id', 'app_domain', 'app_category', 'device_model', 'device_type', 'device_conn_type', 'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21']


In [53]:
hasher = FeatureHasher(numFeatures=10000, inputCols=categorical,
                       outputCol="features")

In [54]:
hasher.transform(df_train).select("features").show()

+--------------------+
|            features|
+--------------------+
|(10000,[437,1228,...|
|(10000,[1228,1289...|
|(10000,[1228,1289...|
|(10000,[1228,1289...|
|(10000,[1228,1289...|
|(10000,[548,1289,...|
|(10000,[548,1289,...|
|(10000,[548,1289,...|
|(10000,[548,1289,...|
|(10000,[548,1289,...|
|(10000,[548,1289,...|
|(10000,[548,1289,...|
|(10000,[548,1289,...|
|(10000,[548,1289,...|
|(10000,[1289,1695...|
|(10000,[1289,1695...|
|(10000,[1289,1695...|
|(10000,[1289,1695...|
|(10000,[1289,1695...|
|(10000,[1289,1695...|
+--------------------+
only showing top 20 rows



In [55]:
classifier = LogisticRegression(maxIter=20, regParam=0.000, elasticNetParam=0.000)

In [56]:
stages = [hasher, classifier]
pipeline = Pipeline(stages=stages)

In [57]:
model = pipeline.fit(df_train)

In [58]:
predictions = model.transform(df_test)

In [59]:
predictions.cache()

DataFrame[label: int, C1: string, banner_pos: string, site_id: string, site_domain: string, site_category: string, app_id: string, app_domain: string, app_category: string, device_model: string, device_type: string, device_conn_type: string, C14: string, C15: string, C16: string, C17: string, C18: string, C19: string, C20: string, C21: string, features: vector, rawPrediction: vector, probability: vector, prediction: double]

In [60]:
ev = BinaryClassificationEvaluator(rawPredictionCol = "rawPrediction", metricName = "areaUnderROC")

In [61]:
print(ev.evaluate(predictions))

0.7428905257649197


In [62]:
spark.stop()

### Feature interaction

In [63]:
from pyspark.ml.feature import RFormula

In [64]:
spark = SparkSession\
    .builder\
    .appName("CTR")\
    .getOrCreate()

In [65]:
schema = StructType([
    StructField("id", StringType(), True),
    StructField("click", IntegerType(), True),
    StructField("hour", IntegerType(), True),
    StructField("C1", StringType(), True),
    StructField("banner_pos", StringType(), True),
    StructField("site_id", StringType(), True),
    StructField("site_domain", StringType(), True),
    StructField("site_category", StringType(), True),
    StructField("app_id", StringType(), True),
    StructField("app_domain", StringType(), True),
    StructField("app_category", StringType(), True),
    StructField("device_id", StringType(), True),
    StructField("device_ip", StringType(), True),
    StructField("device_model", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("device_conn_type", StringType(), True),
    StructField("C14", StringType(), True),
    StructField("C15", StringType(), True),
    StructField("C16", StringType(), True),
    StructField("C17", StringType(), True),
    StructField("C18", StringType(), True),
    StructField("C19", StringType(), True),
    StructField("C20", StringType(), True),
    StructField("C21", StringType(), True),
])

In [66]:
df = spark.read.parquet("./train_parquet/")
df = df.drop('id').drop('hour').drop('device_id').drop('device_ip')
df = df.withColumnRenamed("click", "label")
df = df.sample(fraction=0.1, seed=42)
df_train, df_test = df.randomSplit([0.7, 0.3], 42)
df_train.cache()
df_test.cache()

DataFrame[label: int, C1: string, banner_pos: string, site_id: string, site_domain: string, site_category: string, app_id: string, app_domain: string, app_category: string, device_model: string, device_type: string, device_conn_type: string, C14: string, C15: string, C16: string, C17: string, C18: string, C19: string, C20: string, C21: string]

In [67]:
categorical = df_train.columns
categorical.remove('label')
print(categorical)

['C1', 'banner_pos', 'site_id', 'site_domain', 'site_category', 'app_id', 'app_domain', 'app_category', 'device_model', 'device_type', 'device_conn_type', 'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21']


In [68]:
cat_inter = ['C14', 'C15']
concat = '+'.join(categorical)
interaction = ':'.join(cat_inter)
formula = "label ~ " + concat + '+' + interaction

In [69]:
print(formula)

label ~ C1+banner_pos+site_id+site_domain+site_category+app_id+app_domain+app_category+device_model+device_type+device_conn_type+C14+C15+C16+C17+C18+C19+C20+C21+C14:C15


In [70]:
interactor = RFormula(
    formula=formula,
    featuresCol="features",
    labelCol="label").setHandleInvalid("keep")

In [71]:
interactor.fit(df_train).transform(df_train).select("features").show()

+--------------------+
|            features|
+--------------------+
|(42628,[5,7,2869,...|
|(42628,[5,7,802,3...|
|(42628,[5,7,802,3...|
|(42628,[5,7,802,3...|
|(42628,[5,7,1371,...|
|(42628,[5,7,14,32...|
|(42628,[5,7,14,32...|
|(42628,[5,7,14,32...|
|(42628,[5,7,14,32...|
|(42628,[5,7,14,32...|
|(42628,[5,7,14,32...|
|(42628,[5,7,14,32...|
|(42628,[5,7,14,32...|
|(42628,[5,7,14,32...|
|(42628,[5,7,14,32...|
|(42628,[5,7,14,32...|
|(42628,[5,7,14,32...|
|(42628,[5,7,14,32...|
|(42628,[5,7,14,32...|
|(42628,[5,7,14,32...|
+--------------------+
only showing top 20 rows



In [72]:
classifier = LogisticRegression(maxIter=20, regParam=0.000, elasticNetParam=0.000)

In [73]:
stages = [interactor, classifier]
pipeline = Pipeline(stages=stages)
model = pipeline.fit(df_train)
predictions = model.transform(df_test)
predictions.cache()
predictions.show()

26/06/25 00:46:08 WARN DAGScheduler: Broadcasting large task binary with size 7.4 MiB
26/06/25 00:46:16 WARN DAGScheduler: Broadcasting large task binary with size 7.4 MiB
26/06/25 00:46:16 WARN DAGScheduler: Broadcasting large task binary with size 7.4 MiB
26/06/25 00:46:24 WARN DAGScheduler: Broadcasting large task binary with size 7.4 MiB
26/06/25 00:46:25 WARN DAGScheduler: Broadcasting large task binary with size 7.4 MiB
26/06/25 00:46:25 WARN DAGScheduler: Broadcasting large task binary with size 7.4 MiB
26/06/25 00:46:26 WARN DAGScheduler: Broadcasting large task binary with size 7.4 MiB
26/06/25 00:46:27 WARN DAGScheduler: Broadcasting large task binary with size 7.4 MiB
26/06/25 00:46:27 WARN DAGScheduler: Broadcasting large task binary with size 7.4 MiB
26/06/25 00:46:28 WARN DAGScheduler: Broadcasting large task binary with size 7.4 MiB
26/06/25 00:46:28 WARN DAGScheduler: Broadcasting large task binary with size 7.4 MiB
26/06/25 00:46:29 WARN DAGScheduler: Broadcasting larg

+-----+----+----------+--------+-----------+-------------+--------+----------+------------+------------+-----------+----------------+-----+---+---+----+---+---+------+---+--------------------+--------------------+--------------------+----------+
|label|  C1|banner_pos| site_id|site_domain|site_category|  app_id|app_domain|app_category|device_model|device_type|device_conn_type|  C14|C15|C16| C17|C18|C19|   C20|C21|            features|       rawPrediction|         probability|prediction|
+-----+----+----------+--------+-----------+-------------+--------+----------+------------+------------+-----------+----------------+-----+---+---+----+---+---+------+---+--------------------+--------------------+--------------------+----------+
|    0|1001|         0|51a8ceda|   c4e18dd6|     bcf865d9|ecad2386|  7801e8d9|    07d7df22|    f07e20f8|          1|               2|20153|320| 50|2307|  3|163|    -1| 61|(42628,[5,7,802,3...|[4.35140315574804...|[0.98727529014686...|       0.0|
|    0|1001|    

26/06/25 00:47:08 WARN DAGScheduler: Broadcasting large task binary with size 3.7 MiB


In [74]:
ev = BinaryClassificationEvaluator(rawPredictionCol = "rawPrediction", metricName = "areaUnderROC")

In [75]:
print(ev.evaluate(predictions))

26/06/25 00:47:08 WARN DAGScheduler: Broadcasting large task binary with size 3.7 MiB


0.7449374257827216


In [76]:
spark.stop()